In [ ]:
# =============================================
# 🎵 AI Music Generator (Hearable in Colab)
# =============================================

!pip install pretty_midi music21 tensorflow gradio midi2audio pyfluidsynth > /dev/null
!apt-get install -y fluidsynth ffmpeg > /dev/null
!wget -q https://github.com/FluidSynth/fluidsynth/raw/master/examples/example.sf2 -O /usr/share/sounds/sf2/FluidR3_GM.sf2

import numpy as np
import pretty_midi
from music21 import note, stream
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from midi2audio import FluidSynth
import gradio as gr
import os

# Define note patterns for each style
style_notes = {
    "Classical": ['C4','D4','E4','F4','G4','A4','B4','C5'],
    "Jazz": ['C4','D#4','F4','G4','A#4','C5','D5','G#4'],
    "Pop": ['C4','D4','E4','G4','A4','C5']
}

def build_music_model(input_shape):
    model = Sequential([
        LSTM(128, input_shape=input_shape, return_sequences=True),
        Dropout(0.3),
        LSTM(128),
        Dense(128, activation='relu'),
        Dense(1)
    ])
    model.compile(loss='mse', optimizer='adam')
    return model

def generate_music(style="Classical"):
    # Select notes for chosen style
    notes_list = style_notes[style]
    seq_length = 20
    total_notes = len(notes_list)

    note_to_int = {n:i for i,n in enumerate(notes_list)}
    int_to_note = {i:n for n,i in note_to_int.items()}

    # Random sequences for mini-training
    data = [np.random.randint(0, total_notes, seq_length) for _ in range(200)]
    X = np.array([seq[:-1] for seq in data])
    y = np.array([seq[-1] for seq in data])
    X = X.reshape((X.shape[0], X.shape[1], 1)) / total_notes
    y = y / total_notes

    model = build_music_model((seq_length-1, 1))
    model.fit(X, y, epochs=5, batch_size=16, verbose=0)

    # Generate notes
    pattern = np.random.randint(0, total_notes, seq_length-1).tolist()
    output_notes = []
    for _ in range(50):
        x_pred = np.reshape(pattern, (1, len(pattern), 1)) / total_notes
        prediction = model.predict(x_pred, verbose=0)
        index = int(prediction * total_notes) % total_notes
        result = int_to_note[index]
        output_notes.append(result)
        pattern.append(index)
        pattern = pattern[1:]

    # Convert to MIDI
    midi_stream = stream.Stream()
    for n_str in output_notes:
        n = note.Note(n_str)
        n.quarterLength = 0.5
        midi_stream.append(n)

    midi_filename = f"generated_{style.lower()}.mid"
    midi_stream.write('midi', fp=midi_filename)

    # Convert MIDI to WAV
    sf_path = "/usr/share/sounds/sf2/FluidR3_GM.sf2"
    fs = FluidSynth(sound_font=sf_path)
    wav_file = f"output_{style.lower()}.wav"
    fs.midi_to_audio(midi_filename, wav_file)

    return wav_file


# 🎛️ Build Gradio Interface
def music_app(style):
    wav_path = generate_music(style)
    return wav_path  # Gradio will automatically make it playable

music_styles = ["Classical", "Jazz", "Pop"]

interface = gr.Interface(
    fn=music_app,
    inputs=gr.Dropdown(choices=music_styles, label="🎹 Select Music Style"),
    outputs=gr.Audio(label="🎧 Generated Tune", type="filepath"),
    title="🎼 AI Music Generator",
    description="Select a music style and listen to AI-generated melody."
)

interface.launch(debug=True)


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://6b96c2b0b3b05da828.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)
/tmp/ipython-input-125181998.py:62: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  index = int(prediction * total_notes) % total_notes
/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)
/tmp/ipython-input-125181998.py:62: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecate